# Round 1 ancestry filtering

Fits a Mahalanobis-distance ellipsoid to the EUR reference population cluster in 1000G's own PCA space (`submit_pca_r1.ipynb`'s output), then applies that same fitted ellipsoid to AoU's projected PCs to produce a keep-list. Same method as the earlier `ellopsoid_filter.py` prototype (chi2 threshold, tight reference cluster), now run against the real projected data instead of a chr22 validation subset.

Fit and threshold come from the reference only -- AoU samples never influence the ellipsoid, they're just scored against it. Round 1 uses a deliberately loose threshold (`p=0.999`) against the broad EUR superpop -- round 2 (`round2_filter.ipynb`) refits on CEU/GBR specifically with a tighter, not-yet-decided threshold. (AFR clustering dropped here -- too few reference samples for a reliable ellipsoid fit.)

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

## Inputs

In [ ]:
REF_EIGENVEC_PATH = "/home/jupyter/workspace/Data from All of Us Controlled Tier /shared-env-pilot/data/01_ancestry_filtering/round1_pca/1kg_all_pca_reprojected.sscore"   # 1000G's own reprojected scores, from submit_pca_r1.ipynb (PCi_AVG columns)
REF_PANEL_PATH = "/home/jupyter/workspace/Data from All of Us Controlled Tier /shared-env-pilot/data/integrated_call_samples_v3.20130502.ALL.panel"      # integrated_call_samples_v3.20130502.ALL.panel
AOU_SSCORE_PATH = "/home/jupyter/workspace/Data from All of Us Controlled Tier /shared-env-pilot/data/01_ancestry_filtering/round1_pca/acaf_projected.sscore"     # acaf_projected.sscore, from submit_pca_r1.ipynb
OUT_DIR = "/home/jupyter/workspace/Data from All of Us Controlled Tier /shared-env-pilot/data/01_ancestry_filtering/round1_pca/"             # keep-list + diagnostic plots written here


N_PCS = 2                # matches the validated round-2-prototype parameters
THRESHOLD_QUANTILE = 0.999   # deliberately loose for round 1 -- round 2 refits tighter on CEU/GBR

EUR_REF_POPS = ["CEU", "GBR", "TSI", "FIN", "IBS"]   # reference cluster the ellipsoid is fit to

# all 26 1000G populations, grouped by superpopulation -- used to build POP_COLORS below.
# Not just the reference cluster: the diagnostic plot shows every population in the panel,
# so every population needs a distinguishable color, not just the ones being fit against.
SUPERPOP_POPS = {
    "EUR": ["CEU", "TSI", "FIN", "GBR", "IBS"],
    "AFR": ["YRI", "LWK", "GWD", "MSL", "ESN", "ASW", "ACB"],
    "EAS": ["CHB", "JPT", "CHS", "CDX", "KHV"],
    "SAS": ["GIH", "PJL", "BEB", "STU", "ITU"],
    "AMR": ["MXL", "PUR", "CLM", "PEL"],
}

# one sequential colormap per superpopulation -- continents are grouped by hue (matches the
# convention used in most population-genetics PCA plots), populations within a continent are
# distinguishable shades along that colormap
SUPERPOP_CMAPS = {
    "EUR": "Blues",
    "AFR": "Oranges",
    "EAS": "Greens",
    "SAS": "Purples",
    "AMR": "Reds",
}

# varied per superpopulation rather than one fixed (lo, hi) for everyone -- keeps shades
# from converging toward similar mid-tones across different continents, maximizing overall
# differentiation rather than just within-continent differentiation
SUPERPOP_SHADE_RANGES = {
    "EUR": (0.25, 0.95),
    "AFR": (0.35, 0.90),
    "EAS": (0.30, 0.95),
    "SAS": (0.25, 0.85),
    "AMR": (0.35, 0.90),
}

# cycled per population within each superpopulation -- a second, shape-based channel of
# differentiation on top of color, since two populations in the same continent can still
# land on visually similar shades
POP_MARKERS_CYCLE = ["o", "s", "^", "D", "v", "P", "X", "*"]


def build_pop_colors(superpop_pops=SUPERPOP_POPS, superpop_cmaps=SUPERPOP_CMAPS, shade_ranges=SUPERPOP_SHADE_RANGES):
    import matplotlib
    import matplotlib.colors as mcolors

    colors = {}
    for superpop, pops in superpop_pops.items():
        cmap = matplotlib.colormaps[superpop_cmaps[superpop]]
        lo, hi = shade_ranges[superpop]
        shades = np.linspace(lo, hi, len(pops)) if len(pops) > 1 else [(lo + hi) / 2]
        for pop, frac in zip(pops, shades):
            colors[pop] = mcolors.to_hex(cmap(frac))
    return colors


def build_pop_markers(superpop_pops=SUPERPOP_POPS, markers_cycle=POP_MARKERS_CYCLE):
    markers = {}
    for pops in superpop_pops.values():
        for i, pop in enumerate(pops):
            markers[pop] = markers_cycle[i % len(markers_cycle)]
    return markers


POP_COLORS = build_pop_colors()
POP_MARKERS = build_pop_markers()

os.makedirs(OUT_DIR, exist_ok=True)

## Load reference PCs + population labels

In [ ]:
pc_cols = [f"PC{i}_AVG" for i in range(1, N_PCS + 1)]   # shared by ref and AoU -- both are
                                                          # --score output (no 'sum' modifier),
                                                          # so both use the AVG-scale convention

ref = pd.read_csv(REF_EIGENVEC_PATH, sep=r"\s+")
ref_id_col = "#IID" if "#IID" in ref.columns else "IID"
ref = ref.rename(columns={ref_id_col: "sample"})

labels = pd.read_csv(REF_PANEL_PATH, sep="\t")
ref = ref.merge(labels[["sample", "pop", "super_pop"]], on="sample")

assert all(c in ref.columns for c in pc_cols), f"missing PC columns, have: {list(ref.columns)}"
print("Samples per population:")
print(ref["pop"].value_counts())

## Load AoU projected PCs

Both `AOU_SSCORE_PATH` and `REF_EIGENVEC_PATH` above are `--score` output scored the same way (no `sum` modifier), so they share the same `PCi_AVG` columns and are directly comparable on the same scale -- see `pc_cols` above.

In [ ]:
aou = pd.read_csv(AOU_SSCORE_PATH, sep=r"\s+")
aou_id_col = "#IID" if "#IID" in aou.columns else "IID"
aou = aou.rename(columns={aou_id_col: "sample"})

assert all(c in aou.columns for c in pc_cols), f"missing PC columns, have: {list(aou.columns)}"
print(f"AoU samples: {len(aou)}")

## Fit + validate + score + write + plot

Writes `round1_keep_ids_<cluster>_<prob>.txt` and a diagnostic plot: population pass-rate table on the reference (sanity check before touching AoU), then AoU pass rate against that same fitted ellipsoid. Kept as a function (rather than inlined) in case a second cluster is added back later.

In [ ]:
def mahal(x, mean, cov_inv):
    d = x - mean
    return np.sqrt(d @ cov_inv @ d)


def run_ellipsoid_filter(cluster_name, ref_pop_list):
    ref_mask = ref["pop"].isin(ref_pop_list)
    ref_pcs = ref.loc[ref_mask, pc_cols].values
    assert len(ref_pcs) > N_PCS, f"too few reference samples ({len(ref_pcs)}) to fit a {N_PCS}-PC covariance"

    ref_mean = ref_pcs.mean(axis=0)
    ref_cov = np.cov(ref_pcs, rowvar=False)
    ref_cov_inv = np.linalg.inv(ref_cov)
    threshold = np.sqrt(chi2.ppf(THRESHOLD_QUANTILE, df=N_PCS))
    print(f"[{cluster_name}] Mahalanobis threshold ({N_PCS} PCs, {THRESHOLD_QUANTILE:.0%}): {threshold:.3f}")

    # e.g. 0.9 -> "p90" -- tags output filenames with the threshold quantile used, so
    # results from different THRESHOLD_QUANTILE runs don't silently overwrite each other
    prob_tag = f"p{THRESHOLD_QUANTILE * 100:g}"

    mahal_col, pass_col = f"mahal_{cluster_name}", f"pass_{cluster_name}"

    # validate against the full reference panel before touching AoU
    ref[mahal_col] = [mahal(row, ref_mean, ref_cov_inv) for row in ref[pc_cols].values]
    ref[pass_col] = ref[mahal_col] <= threshold

    print(f"[{cluster_name}] Pass rate by population:")
    summary = ref.groupby("pop")[pass_col].agg(["sum", "count"])
    summary["pct"] = (summary["sum"] / summary["count"] * 100).round(1)
    print(summary.sort_values("pct", ascending=False))

    ref_retention = ref.loc[ref_mask, pass_col].mean()
    print(f"[{cluster_name}] {'+'.join(ref_pop_list)} retention: {ref_retention:.1%}")

    # score AoU against the same fitted ellipsoid
    aou[mahal_col] = [mahal(row, ref_mean, ref_cov_inv) for row in aou[pc_cols].values]
    aou[pass_col] = aou[mahal_col] <= threshold
    n_pass = aou[pass_col].sum()
    print(f"[{cluster_name}] AoU passing: {n_pass} ({n_pass / len(aou):.1%})")

    # keep-list -- AoU person IDs only, participant-level, stays in the bucket, never committed to git
    keep_path = os.path.join(OUT_DIR, f"round1_keep_ids_{cluster_name}_{prob_tag}.txt")
    aou.loc[aou[pass_col], "sample"].to_csv(keep_path, index=False, header=False)
    print(f"[{cluster_name}] Wrote {n_pass} IDs to {keep_path}")

    # diagnostics -- PC1 vs PC2 with the fitted ellipse, reference vs AoU overlaid;
    # Mahalanobis distance distributions. Plot saved under OUT_DIR, not committed.
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # AoU drawn first (background, grey, behind), reference drawn second (foreground,
    # full alpha, on top) -- reference is the labeled/interpretable set and should read
    # clearly over the AoU cloud, not be buried under it
    axes[0].scatter(
        aou[pc_cols[0]], aou[pc_cols[1]],
        c=np.where(aou[pass_col], "black", "grey"),
        marker="x", alpha=0.3, s=10, label="AoU",
    )
    for pop, grp in ref.groupby("pop"):
        axes[0].scatter(
            grp[pc_cols[0]], grp[pc_cols[1]],
            c=POP_COLORS.get(pop, "grey"), marker=POP_MARKERS.get(pop, "o"),
            label=pop, alpha=1.0, s=15,
        )

    cov_2d = np.cov(ref_pcs[:, :2], rowvar=False)
    mean_2d = ref_pcs[:, :2].mean(axis=0)
    eigvals, eigvecs = np.linalg.eigh(cov_2d)
    angle = np.degrees(np.arctan2(*eigvecs[:, 1][::-1]))
    scale_2d = np.sqrt(chi2.ppf(THRESHOLD_QUANTILE, df=2))
    width = 2 * scale_2d * np.sqrt(eigvals[1])
    height = 2 * scale_2d * np.sqrt(eigvals[0])
    ellipse = Ellipse(
        xy=mean_2d, width=width, height=height, angle=angle,
        fill=False, edgecolor="black", linewidth=2, linestyle="--",
        label=f"{THRESHOLD_QUANTILE:.0%} {'+'.join(ref_pop_list)} ellipsoid (2D slice)",
    )
    axes[0].add_patch(ellipse)
    axes[0].set_xlabel(pc_cols[0])
    axes[0].set_ylabel(pc_cols[1])
    axes[0].set_title(f"{pc_cols[0]} vs {pc_cols[1]} -- {cluster_name} reference + AoU")
    axes[0].legend(markerscale=2, fontsize=8)

    for pop, grp in ref.groupby("pop"):
        axes[1].hist(grp[mahal_col], bins=30, alpha=0.5, label=pop, color=POP_COLORS.get(pop, "grey"), density=True)
    axes[1].hist(aou[mahal_col], bins=30, alpha=0.4, label="AoU", color="royalblue", density=True)
    axes[1].axvline(threshold, color="black", linestyle="--", linewidth=2, label=f"Threshold ({threshold:.2f})")
    axes[1].set_xlabel(f"Mahalanobis distance from {'+'.join(ref_pop_list)} centroid ({N_PCS} PCs)")
    axes[1].set_ylabel("Density")
    axes[1].legend(fontsize=8)

    plt.suptitle(
        f"Round 1 ancestry filter [{cluster_name}] -- {'+'.join(ref_pop_list)} retention: {ref_retention:.1%}  |  "
        f"AoU pass rate: {n_pass / len(aou):.1%}",
        fontsize=11,
    )
    plt.tight_layout()
    plot_path = os.path.join(OUT_DIR, f"round1_filter_{cluster_name}_{prob_tag}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[{cluster_name}] Plot saved to {plot_path}")

    return {"mean": ref_mean, "cov_inv": ref_cov_inv, "threshold": threshold, "keep_path": keep_path}

### EUR-like cluster

In [ ]:
eur_result = run_ellipsoid_filter("eur", EUR_REF_POPS)